In [2]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.amc.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
amc_id = '41_1'
path = r"C:\Users\kaustubh.keny\Downloads\Factsheet 09-02-2026\41_31-Jan-26_1_FS.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object =UTIPassive(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
# print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2026\41_1_AMC.json5


In [ ]:
# len(title)
title

In [29]:
config = get_config("2026",amc_id)
regex = get_regex("2026")
object = UTIPassive(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2026\41_1_AMC.json5


In [31]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\41_31-Jan-26_1_FS.json


In [ ]:
pattern = "(?:Mr\\.?|Mrs\\.?|Ms\\.?)\\s*([A-Za-z]+\\s*[A-Za-z]+\\s*(?:Patil|Goyal)?).*?Managing (?:the|this) scheme since\\s*([A-Za-z]+\\s*\\-?\\s*[0-9]+|Inception|[0-9]+-[A-Za-z]+-[0-9]+).*?Total\\s*Exp\\:?\\s*(\\d+\\s*(?:[Yy]rs?|[Yy]ears?))"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("manager"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

UTI NIFTY 50 INDEX FUND
Mr. Sharwan Kumar Goyal, B.Com, CFA, MMS Managing the scheme since July 2018 Total Exp: 19 Yrs Mr Ayush Jain, Assistant Fund Manager CA, B.Com (Tax) Managing the scheme since May 2022 Total Exp: 7 Yrs
[('Sharwan Kumar Goyal', 'July 2018', '19 Yrs'), ('Ayush Jain', 'May 2022', '7 Yrs')]
UTI NIFTY NEXT 50 INDEX FUND
Mr. Sharwan Kumar Goyal, B.Com, CFA, MMS Managing the scheme since June 2018. Total Exp: 19 Yrs Mr Ayush Jain, Assistant Fund Manager CA, B.Com (Tax) Managing the scheme since May 2022 Total Exp: 7 Yrs
[('Sharwan Kumar Goyal', 'June 2018', '19 Yrs'), ('Ayush Jain', 'May 2022', '7 Yrs')]
UTI NIFTY 200 MOMENTUM 30 INDEX FUND
Mr. Sharwan Kumar Goyal, B.Com,CFA, MMS Managing the scheme since Mar 2021. Total Exp: 19 Yrs Mr Ayush Jain, Assistant Fund Manager CA, B.Com (Tax) Managing the scheme since May 2022 Total Exp: 7 Yrs
[('Sharwan Kumar Goyal', 'Mar 2021', '19 Yrs'), ('Ayush Jain', 'May 2022', '7 Yrs')]
UTI BSE SENSEX INDEX FUND
Mr. Sharwan Kumar Goyal,

In [ ]:
import json, csv, unicodedata, os, ast, re
import pandas as pd
from pathlib import Path
from datetime import datetime

def sid_to_csv(json_path, output_folder=""):
    json_path = Path(json_path)

    output_folder = Path(output_folder) if output_folder else json_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    csv_path = output_folder / f"{json_path.stem}.csv"

    
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    record = doc.get("value", {}) or {}

    def write_df(fh, df):
        df.to_csv(fh, index=False)
        fh.write("\n\n")  # 2 blank rows as section separator

    with open(csv_path, "w", encoding="utf-8", newline="") as fh:

        kv_rows = []
        for k in sorted(record.keys()):
            if k not in ("fund_manager", "load","field_location"):
                v = record.get(k)
                # print(v)
                if isinstance(v, (dict,list)):
                    v = " ".join(v)
                
                kv_rows.append({
                    "key": k,
                    "value": v if v is not None else ""
                })
            
            if k == "field_location":
                v = record.get(k)
                v = str(v)
                kv_rows.append(
                     "key": k,
                     "value": v if v is not None else ""
                )

        df1 = pd.DataFrame(kv_rows, columns=["key", "value"])
        write_df(fh, df1)

        fund_manager = record.get("fund_manager") or []
        if fund_manager:
            df2 = pd.DataFrame(fund_manager)
            write_df(fh, df2)

        load = record.get("load") or []
        if load:
            df3 = pd.DataFrame(load)
            write_df(fh, df3)

    return str(csv_path)

def csv_to_sid_json(csv_path, output_folder=""):
    csv_path = Path(csv_path)

    output_folder = Path(output_folder) if output_folder else csv_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    json_path = output_folder / f"{csv_path.stem}.json"

    # ---------- split CSV into sections ----------
    sections = []
    current = []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not any(cell.strip() for cell in row):
                if current:
                    sections.append(current)
                    current = []
            else:
                current.append(row)

        if current:
            sections.append(current)

    value = {}
    if sections:
        header, *rows = sections[0]
        key_idx = header.index("key")
        val_idx = header.index("value")

        for r in rows:
            k = r[key_idx].strip()
            v = r[val_idx].strip()

            if k == "field_location" and v:
                try:
                    v = ast.literal_eval(v)  # safe dict restore
                except Exception:
                    pass

            value[k] = v

    if len(sections) > 1:
        header, *rows = sections[1]
        fund_manager = []

        for r in rows:
            rec = {
                h: r[i].strip() if i < len(r) else ""
                for i, h in enumerate(header)
                if h
            }
            fund_manager.append(rec)

        value["fund_manager"] = fund_manager

    if len(sections) > 2:
        header, *rows = sections[2]
        load = []

        for r in rows:
            rec = {
                h: r[i].strip() if i < len(r) else ""
                for i, h in enumerate(header)
                if h
            }
            load.append(rec)

        value["load"] = load

    # ---------- METADATA ----------
    metadata = {
        "document_name": csv_path.name,
        "file_type": "sid",
        "process_date": datetime.today().strftime("%Y%m%d")
    }

    # ---------- FINAL JSON ----------
    output = {
        "metadata": metadata,
        "value": value
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    return str(json_path)

def kim_to_csv(json_path, output_folder=""):
    json_path = Path(json_path)

    output_folder = Path(output_folder) if output_folder else json_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    csv_path = output_folder / f"{json_path.stem}.csv"

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    record = (
        doc.get("records", [{}])[0]
        .get("value", {})
    ) or {}

    def write_df(fh, df):
        df.to_csv(fh, index=False)
        fh.write("\n\n")  # section separator

    with open(csv_path, "w", encoding="utf-8", newline="") as fh:

        # ================= DF1: Static KV =================
        kim_static_keys = [
            "amc_name",
            "main_scheme_name",
            "mutual_fund_name",
            "description",
            "field_location"   # ✅ added
        ]

        kv_rows = []
        for k in kim_static_keys:
            v = record.get(k, "")
            if isinstance(v, (dict, list)):
                v = str(v)
            kv_rows.append({
                "key": k,
                "value": v if v is not None else ""
            })

        df1 = pd.DataFrame(kv_rows, columns=["key", "value"])
        write_df(fh, df1)

        # ================= DF2: Asset Allocation =================
        allocation_rows = []

        for item in record.get("asset_allocation_pattern", []):
            row = {
                "instrument_type": item.get("instrument_type", ""),
                "risk_profile": item.get("risk_profile", "")
            }

            row.update({"min": "", "max": "", "total": ""})

            for alloc in item.get("allocation", []):
                alloc_type = alloc.get("type")
                if alloc_type in row:
                    row[alloc_type] = alloc.get("value", "")

            allocation_rows.append(row)

        if allocation_rows:
            df2 = pd.DataFrame(
                allocation_rows,
                columns=["instrument_type", "min", "max", "total", "risk_profile"]
            )
            write_df(fh, df2)

    return str(csv_path)

def csv_to_kim_json(csv_path, output_folder=""):
    csv_path = Path(csv_path)

    output_folder = Path(output_folder) if output_folder else csv_path.parent
    output_folder.mkdir(parents=True, exist_ok=True)

    json_path = output_folder / f"{csv_path.stem}.json"

    # ---------- split CSV into sections ----------
    sections, current = [], []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not any(cell.strip() for cell in row):
                if current:
                    sections.append(current)
                    current = []
            else:
                current.append(row)
        if current:
            sections.append(current)

    value = {}
    if sections:
        header, *rows = sections[0]
        key_idx = header.index("key")
        val_idx = header.index("value")

        for r in rows:
            k = r[key_idx].strip()
            v = r[val_idx].strip()

            if k == "field_location" and v:
                try:
                    v = ast.literal_eval(v)  # ✅ safe restore
                except Exception:
                    v = []

            value[k] = v

    if len(sections) > 1:
        header, *rows = sections[1]
        asset_allocation_pattern = []

        for r in rows:
            row = {h: r[i].strip() if i < len(r) else "" for i, h in enumerate(header)}

            allocation = [
                {"type": "min", "value": row.get("min", "")},
                {"type": "max", "value": row.get("max", "")},
                {"type": "total", "value": row.get("total", "")},
            ]

            asset_allocation_pattern.append({
                "instrument_type": row.get("instrument_type", ""),
                "risk_profile": row.get("risk_profile", ""),
                "allocation": allocation
            })

        value["asset_allocation_pattern"] = asset_allocation_pattern

    # ================= METADATA =================
    metadata = {
        "document_name": csv_path.name,
        "file_type": "kim",
        "process_date": datetime.today().strftime("%Y%m%d")
    }

    output = {
        "metadata": metadata,
        "records": [{"value": value}]
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    return json_path
    
path = r"C:\Users\kaustubh.keny\Downloads\100_18011_Jan-2026_1769580239_SID.json"
k_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\notebook\fs_sidkim\27_18016_Jan-2026_1769757893_KIM.json"
str_path = sid_to_csv(path)
csv_to_sid_json(str_path)
# str_path = kim_to_csv(k_path)
# print(str_path)
# csv_to_kim_json(str_path)


Wealth Company Asset Management Holdings Private Limited
CRISIL Hybrid 50+50 - Moderate Index
on: February 10, 2026
Rs.10
[{'benchmark_index': 3, 'close_date': 1, 'fund_manager': 0, 'load': 3, 'main_scheme_name': 3, 'min_addl_amt': 3, 'min_addl_amt_multiple': 3, 'min_amt': 3, 'min_amt_multiple': 3, 'offer_price': 3, 'open_date': 1, 'scheme_code': 3, 'scheme_objective': 3, 'suitable_for_investors': 1, 'type_of_scheme': 3}]


TypeError: sequence item 0: expected str instance, dict found

In [ ]:
#bajaj
# def _update_bench_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["benchmark"],data,re.IGNORECASE)
#     return {"benchmark_index":matches[0] if matches else ""}

# def _update_date_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["date"],data,re.IGNORECASE)
#     return {"scheme_launch_date":matches[0] if matches else ""}
 

#generic
def _update_benchmark_data(self,main_key:str,bench_data):
    bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
    bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
    if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
        return {"benchmark_index":match[0]}
    return {main_key:bench_data}
   

def _update_date_data(self,main_key:str,data):
    date_data = " ".join(data) if isinstance(data, list) else data
    if match := re.findall(self.REGEX["date"],date_data,re.IGNORECASE):
        return {"scheme_launch_date":match[0]}
    return {main_key:date_data}

   
#canara
# def _update_benchmark_data(self,main_key:str,bench_data):
#     bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
#     if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
#         return {main_key:match[0]}
#     return {main_key:bench_data}

#dsp
# def _update_benchmark_data(self,main_key:str,bench_data):
#     bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
#     if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
#         return {main_key:match[0]}
#     return {main_key:bench_data}

# def _update_date_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["date"],data,re.IGNORECASE)
#     return {"scheme_launch_date":matches[0] if matches else ""}

#hdfc
# def _update_date_data(self,main_key:str,data):
#     if matches:=re.findall(self.REGEX["date"],data, re.IGNORECASE):
#         return {main_key:matches[0]}
def _update_benchmark_data(self,main_key:str,data):
    data=re.sub(self.REGEX["benchmark"],"", data, re.IGNORECASE).strip()
    if matches:= re.findall(self.REGEX["benchmark2"],data,re.IGNORECASE):
        return {main_key:matches[0]}
    return {main_key:data}

#navi
# def _extract_benchmark_data(self,main_key:str,data:str,pattern:str):
#     bench_data = f"{main_key} {data}"
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data).strip()
#     if matches:=re.findall(self.REGEX[pattern],bench_data, re.IGNORECASE):
#         return {"benchmark_index":matches[0]}
#     return{"benchmark_index":f"{main_key} {data}"}


           
def _extract_bench_data(self,main_key:str,data,pattern:str):
    data = " ".join(data) if isinstance(data,list) else data
    data = re.sub(r"Ni\s*y","Nifty",data, re.IGNORECASE)
    return {main_key:data}
    

# def _update_date_data(self,main_key:str,data):
#     if matches:=re.findall(self.REGEX["date"],data, re.IGNORECASE):
#         return {main_key:matches[0]}

# def _update_date_data(self, main_key:str,data):  # GROWW & Edelweiss
#     date_data = " ".join(data) if isinstance(data,list) else data
#     matches = re.findall(self.REGEX["date"],date_data, re.IGNORECASE)
#     return {"scheme_launch_date": " ".join(matches)}

In [ ]:
import os
import json
import pandas as pd


# ---------------- FIELD CONFIG (converted from your JSON) ---------------- #

FIELD_KEYS = {
    "load_keys": ["entry", "exit"],

    "manager_keys": [
        "name",
        "managing_fund_since",
        "total_exp",
        "qualification"
    ],

    "metric_keys": [
        "alpha",
        "arithmetic_mean_ratio",
        "average_div_yield",
        "average_pb",
        "average_pe",
        "avg_maturity",
        "beta",
        "correlation_ratio",
        "downside_deviation",
        "information_ratio",
        "macaulay",
        "mod_duration",
        "port_turnover_ratio",
        "r_squared_ratio",
        "roe_ratio",
        "sharpe",
        "sortino_ratio",
        "std_dev",
        "tracking_error",
        "treynor_ratio",
        "upside_deviation",
        "ytm"
    ],

    "static_keys": [
        "amc_name",
        "main_scheme_name",
        "mutual_fund_name",
        "benchmark_index",
        "monthly_aaum_date",
        "monthly_aaum_value",
        "scheme_launch_date",
        "min_addl_amt",
        "min_addl_amt_multiple",
        "min_amt",
        "min_amt_multiple",
        "Riskometer_benchmark",
        "Riskometer"
    ],

    "field_location": "field_location"
}


# ---------------- MAIN CONVERTER ---------------- #

def json_to_csv(json_path):

    os.makedirs(output_dir, exist_ok=True)

    static_keys = FIELD_KEYS["static_keys"]
    load_keys = FIELD_KEYS["load_keys"]
    metric_keys = FIELD_KEYS["metric_keys"]
    manager_keys = FIELD_KEYS["manager_keys"]
    field_location = FIELD_KEYS["field_location"]

    EXCLUDE_KEYS = {"Riskometer", "Riskometer_benchmark", "Riskometer_scheme"}

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    records = doc.get("records", [])

    max_manager = max((len(r["value"].get("fund_manager", [])) for r in records), default=1)

    # headers
    headers = [k for k in static_keys if k not in EXCLUDE_KEYS] + load_keys + metric_keys

    for i in range(1, max_manager + 1):
        headers.extend([f"{k}_{i}" for k in manager_keys])

    headers.append(field_location)

    def flatten(value):

        for bad in EXCLUDE_KEYS:
            value.pop(bad, None)

        row = []

        # static
        for k in static_keys:
            if k in EXCLUDE_KEYS:
                continue

            v = value.get(k, "")
            if isinstance(v, list):
                v = ", ".join(map(str, v))
            row.append(v)

        # load
        entry = ""
        exit_ = ""

        for l in value.get("load", []):
            if l.get("type") == "entry":
                entry = l.get("comment", "")
            elif l.get("type") == "exit":
                exit_ = l.get("comment", "")

        row.extend([entry, exit_])

        # metrics
        metric_map = {m.get("name"): m.get("value") for m in value.get("metrics", [])}
        row.extend([metric_map.get(k, "") for k in metric_keys])

        # managers
        managers = value.get("fund_manager", [])

        for i in range(max_manager):
            if i < len(managers):
                fm = managers[i]
                row.extend([fm.get(k, "") for k in manager_keys])
            else:
                row.extend([""] * len(manager_keys))

        # field location
        fl = value.get("field_location", [{}])
        if isinstance(fl, list) and fl:
            fl_val = json.dumps(fl[0], ensure_ascii=False)
        else:
            fl_val = ""

        row.append(fl_val)

        return row

    rows = [flatten(r["value"]) for r in records]

    for i, r in enumerate(rows[:3]):
        if len(r) != len(headers):
            raise ValueError("Header / row mismatch")

    base = os.path.splitext(os.path.basename(json_path))[0]
  
    pd.DataFrame(rows, columns=headers).to_csv(f"{base}.csv", index=False, encoding="utf-8")

    return f"{base}.csv"


# ---------------- SIMPLE ATTACH FUNCTION ---------------- #

json_file = r"C:\Users\rando\Office Projects\mywork-repo\notebook\fs_sidkim\6_31-Aug-25_FS.json"

def convert_json_file(json_file):
    return json_to_csv(json_file)


convert_json_file(json_file)

'csv_folder\\6_31-Aug-25_FS.csv'

In [11]:
import os, json
import pandas as pd


"Helios Capital Asset Management  (India) Private Limited"

static_keys = [
        "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index", "monthly_aaum_date", 
        "monthly_aaum_value", "scheme_launch_date", "min_addl_amt", "min_addl_amt_multiple", 
        "min_amt", "min_amt_multiple",
    ]
load_keys = ["entry","exit"]
metric_keys = [
    "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe", "avg_maturity",
    "beta", "correlation_ratio", "downside_deviation", "information_ratio", "macaulay",
    "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio", "sharpe", "sortino_ratio",
    "std_dev", "tracking_error", "treynor_ratio", "upside_deviation", "ytm"
]

manager_keys = ["name","managing_fund_since","total_exp","qualification"]

def flatten_to_row(value, max_manager):
    add_value = []

    # Static keys
    for k in static_keys:
        val = value.get(k, "")
        if isinstance(val, list):
            val = ", ".join(val)
        add_value.append(val)

    # Loads
    entry, exit = "", ""
    for l in value.get("load", []):
        if l.get("type") == "entry":
            entry = l.get("comment", "")
        elif l.get("type") == "exit":
            exit = l.get("comment", "")
    add_value.extend([entry, exit])

    # Metrics
    metric_map = {m["name"]: m["value"] for m in value.get("metrics", [])}
    metric = [metric_map.get(k, "") for k in metric_keys]
    add_value.extend(metric)

    # Fund managers (pad to max_manager)
    managers = value.get("fund_manager", [])
    for i in range(max_manager):
        if i < len(managers):
            fm = managers[i]
            add_value.extend([
                fm.get("name", ""),
                fm.get("managing_fund_since", ""),
                fm.get("total_exp", ""),
                fm.get("qualification", "")
            ])
        else:
            # pad with blanks if fewer managers
            add_value.extend(["", "", "", ""])

    return add_value


path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\8_31-Dec-25_FS.json"
with open(path,"r",encoding="utf8") as f:
    df = json.load(f)
sheet_name = df.get("metadata",{}).get("document_name","")
records = df.get("records",[])
max_fund_managers = max(
    (len(record["value"].get("fund_manager", [])) for record in records),
    default=0
)


headers = static_keys + load_keys + metric_keys
for i in range(1, max_fund_managers + 1):
    headers.extend([f"{key}_{i}" for key in manager_keys])

file_name = path.split("\\")[-1].replace(".json","")

rows = [flatten_to_row(record["value"], max_fund_managers) for record in records]
df_out = pd.DataFrame(rows, columns=headers)
df_out.to_csv(f"{file_name}.csv", index=False)



In [ ]:
import pandas as pd
import re, os

path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\amfii-website\VAHAN_DATA_2025.csv"
df = pd.read_csv(path)
# df.head(10)

months = [f"{m},2025" for m in 
          ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]]

pivot = df.assign(val="1") \
          .pivot_table(index=["STATE","RTO"],
                       columns="DATE",
                       values="val",
                       aggfunc="first") \
          .reindex(columns=months, fill_value="NA")

result = pivot.reset_index()
result.to_csv("validate_2025.csv", index=False)

In [33]:
def json_to_csv(json_path, output_dir="."):
    keys = {
            "static_keys": [
                "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index",
                "monthly_aaum_date", "monthly_aaum_value", "scheme_launch_date",
                "min_addl_amt", "min_addl_amt_multiple", "min_amt", "min_amt_multiple"
            ],
            "load_keys": ["entry", "exit"],
            "metric_keys": [
                "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe",
                "avg_maturity", "beta", "correlation_ratio", "downside_deviation", "information_ratio",
                "macaulay", "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio",
                "sharpe", "sortino_ratio", "std_dev", "tracking_error", "treynor_ratio",
                "upside_deviation", "ytm"
            ],
            "manager_keys": ["name", "managing_fund_since", "total_exp", "qualification"]
        }
    
    static_keys, load_keys, metric_keys, manager_keys = ( keys["static_keys"], keys["load_keys"], keys["metric_keys"], keys["manager_keys"] )

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    records = doc.get("records", [])
    max_manager = max((len(r["value"].get("fund_manager", [])) for r in records), default=1)

    # Build headers
    headers = static_keys + metric_keys
    for i in range(1, max_manager + 1):
        headers.extend([f"{k}_{i}" for k in manager_keys])
    headers.extend(load_keys)

    def flatten_to_row(value):
        row = []
        # static
        for k in static_keys:
            v = value.get(k, "")
            if isinstance(v, list):
                v = ", ".join(v)
            row.append(v)
        # metrics
        metric_map = {m.get("name"): m.get("value") for m in value.get("metrics", [])}
        row.extend([metric_map.get(k, "") for k in metric_keys])
        # fund managers
        managers = value.get("fund_manager", [])
        for i in range(max_manager):
            if i < len(managers):
                fm = managers[i]
                row.extend([fm.get(k, "") for k in manager_keys])
            else:
                row.extend([""] * len(manager_keys))
        
        # loads
        entry, exit_ = "", ""
        for l in value.get("load", []):
            if l.get("type") == "entry": entry = l.get("comment", "")
            elif l.get("type") == "exit": exit_ = l.get("comment", "")
        row.extend([entry, exit_])
        
        return row

    rows = [flatten_to_row(r["value"]) for r in records]

    # Timestamped filename
    base = os.path.splitext(os.path.basename(json_path))[0]
    csv_path = os.path.join(output_dir, f"{base}.csv")
    pd.DataFrame(rows, columns=headers).to_csv(csv_path, index=False, encoding="utf-8")
    return csv_path

path = r"C:\Users\kaustubh.keny\Downloads\71_31-Dec-25_FS.json"
json_to_csv(path)

'.\\71_31-Dec-25_FS.csv'